# ResNet-18 Image Classification with Hugging Face Transformers

이 노트북은 Kaggle Note 환경에서 Hugging Face의 `microsoft/resnet-18` 모델을 사용해 이미지를 분류하는 예제입니다.

구성:
- `datasets` 라이브러리로 샘플 이미지 로드
- `AutoImageProcessor`로 전처리
- PyTorch 기반 추론 수행
- 가장 높은 확률의 클래스 출력
- Top-10 예측 결과 출력

In [ ]:
# Kaggle 환경에서 필요한 라이브러리가 없을 수 있으므로 설치합니다.
# 이미 설치되어 있다면 빠르게 지나갑니다.
!pip -q install datasets transformers

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, ResNetForImageClassification
from IPython.display import display

# 실행 장치를 설정합니다. Kaggle GPU가 켜져 있으면 자동으로 CUDA를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# datasets 라이브러리에서 샘플 이미지를 불러옵니다.
# Hugging Face에서 자주 사용하는 예제 이미지 데이터셋입니다.
dataset = load_dataset("huggingface/cats-image", split="test")
sample = dataset[0]
image = sample["image"]

display(image)

In [ ]:
# 사전학습된 ResNet-18 모델과 이미지 전처리기를 불러옵니다.
model_name = "microsoft/resnet-18"
processor = AutoImageProcessor.from_pretrained(model_name)
model = ResNetForImageClassification.from_pretrained(model_name).to(device)
model.eval()

In [ ]:
# AutoImageProcessor를 이용해 모델 입력 형태로 전처리합니다.
inputs = processor(images=image, return_tensors="pt")
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)

# 가장 높은 확률의 클래스(top-1)를 구합니다.
top1_index = probabilities.argmax(dim=-1).item()
top1_label = model.config.id2label[top1_index]
top1_score = probabilities[0, top1_index].item()

print(f"Top-1 예측 클래스: {top1_label}")
print(f"Top-1 확률: {top1_score:.4%}")

In [ ]:
# top-10 예측 결과를 확률 순으로 출력합니다.
topk = 10
top_probs, top_indices = torch.topk(probabilities, k=topk, dim=-1)

print("Top-10 예측 결과")
print("-" * 60)

for rank, (prob, idx) in enumerate(zip(top_probs[0], top_indices[0]), start=1):
    class_id = idx.item()
    class_name = model.config.id2label[class_id]
    print(f"{rank:>2}. {class_name:<30} {prob.item():.4%}")